# 【notebook｜N-14】RLVR 可验证奖励
> 对应：《06-RL-RLVR/01-RL与RLVR原理.md》【文档｜DOC-RL】；代码 `F-b06-grpo`

- 理论：`06-RL-RLVR/01-RL与RLVR原理.md` §8/§9
- 源码：`nanoproof/nanoproof/rl.py:983-1003`
- Colab：CPU 即可（仅 torch）。

## 理论回顾

RLVR 奖励来自确定性验证器 $r=\mathbb{1}[\mathrm{verifier}=\mathrm{pass}]$。本 notebook 比较稀疏/形状奖励，并复刻 unlikelihood 负样本损失 $-\log(1-p)$。任务：输入 $(a,b)$ 输出 $(a+b)\bmod 10$。

In [1]:
# 【N-14.c1】
import torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); gen = torch.Generator().manual_seed(0)
def v_exact(p,t): return 1.0 if int(p)==int(t) else 0.0
def v_shaped(p,t): return 1.0 if int(p)==int(t) else -abs(int(p)-int(t))/10.0
def v_majority(xs): v=[int(x) for x in xs]; return max(set(v), key=v.count)
print('device', device)
print([v_exact(7,7), v_shaped(6,7), v_majority([3,3,4,3,5])])

device cuda
[1.0, -0.1, 3]


In [2]:
# 【N-14.c2】
class Policy(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Linear(20,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x): return self.net(x)
def onehot(B):
    a=torch.randint(0,10,(B,),generator=gen); b=torch.randint(0,10,(B,),generator=gen)
    x=torch.zeros(B,20); x[torch.arange(B),a]=1; x[torch.arange(B),10+b]=1
    return x.to(device), a.to(device), b.to(device)
def train(rf, steps=400, lr=3e-3):
    net=Policy().to(device); opt=torch.optim.Adam(net.parameters(),lr=lr)
    for _ in range(steps):
        x,a,b=onehot(64); logp=F.log_softmax(net(x),-1)
        pred=torch.multinomial(logp.exp(),1).squeeze(-1); truth=(a+b)%10
        r=torch.tensor([rf(int(p),int(t)) for p,t in zip(pred,truth)],device=device)
        if float(r.std())<1e-6: continue
        adv=(r-r.mean())/(r.std()+1e-6)
        loss=-(logp.gather(-1,pred.unsqueeze(-1)).squeeze(-1)*adv).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x,a,b=onehot(500); return float((net(x).argmax(-1)==(a+b)%10).float().mean())
print('sparse:', round(train(v_exact),3), '| shaped:', round(train(v_shaped),3))

sparse: 0.71 | shaped: 0.368


In [3]:
# 【N-14.c3】
logits=torch.randn(4,3); pred=torch.tensor([0,1,2,0]); logp=F.log_softmax(logits,-1)
ce=-logp.gather(-1,pred.unsqueeze(-1)).squeeze(-1)
p=torch.exp(-ce).clamp(max=1-1e-6); u=-torch.log1p(-p)
print('CE =',[round(v,3) for v in ce.tolist()])
print('unlikelihood =',[round(v,3) for v in u.tolist()])

CE = [2.968, 1.03, 1.539, 0.181]
unlikelihood = [0.053, 0.442, 0.241, 1.796]


## 思考题

1. 稀疏奖励全 0 组为何无梯度？形状奖励如何缓解？
2. `-log(1-p)` 在 $p\to1$ 时如何？为何截断？
3. majority-vote 伪标签何时放大错误？
4. RLVR 为何只重分配 pass@k？

## 预期输出
device；验证器 `[1.0, -0.1, 3]`；两种奖励 accuracy 高于 10%；unlikelihood 随 CE 单调增。